In [1]:
import csv
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import glob

RANDOM_SEED = 42

# Dataset Path

In [2]:
model_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.keras'
tflite_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.tflite'

# Parameters

In [3]:
SEQUENCE_LENGTH = 25
FEATURES_PER_FRAME = 80  # 42 hand + 10 face + 8 pose + 20 relative

# Load Dataset

In [4]:
# Load Dataset
csv_files = sorted(glob.glob('Words-Dataset/*_sequence.csv'))
X_sequences = []
y_sequences = []
for csv_file in csv_files:
    data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
    X_sequences.append(data[:, 1:].reshape(-1, SEQUENCE_LENGTH, FEATURES_PER_FRAME))
    y_sequences.extend(data[:, 0])

X_dataset = np.concatenate(X_sequences, axis=0)
y_dataset = np.array(y_sequences)
y_dataset -= 1  # Convert from 1-based to 0-based indexing for TensorFlow

X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

# Load labels to determine number of classes
with open('Word-Label/keypoint_sequence_classifier_label.csv', encoding='utf-8-sig') as f:
    keypoint_sequence_classifier_labels = csv.reader(f)
    keypoint_sequence_classifier_labels = [row[0] for row in keypoint_sequence_classifier_labels]
NUM_CLASSES = len(keypoint_sequence_classifier_labels)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Labels: {keypoint_sequence_classifier_labels}")

Number of classes: 6
Labels: ['全部', '---', '多謝', '失望', '來', 'byebye']


# Build LSTM Model

In [5]:
if 'NUM_CLASSES' not in locals() or NUM_CLASSES == 0:
    print("No classes found. Please collect data first.")
else:
    model = tf.keras.models.Sequential([
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True), input_shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(256, return_sequences=True)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirection  (None, 25, 512)           690176    
 al)                                                             
                                                                 
 bidirectional_1 (Bidirecti  (None, 25, 512)           1574912   
 onal)                                                           
                                                                 
 dropout (Dropout)           (None, 25, 512)           0         
                                                                 
 bidirectional_2 (Bidirecti  (None, 256)               656384    
 onal)                                                           
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                       

# Compile and Train Model

In [6]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cp_callback = tf.keras.callbacks.ModelCheckpoint(model_save_path, verbose=1, save_weights_only=False)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

model.fit(X_train, y_train, epochs=2000, batch_size=32, validation_data=(X_test, y_test), callbacks=[cp_callback, es_callback])


Epoch 1/2000


112/112 [==============================] - ETA: 0s - loss: 0.3111 - accuracy: 0.8840
Epoch 1: saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
112/112 [==============================] - 26s 166ms/step - loss: 0.3111 - accuracy: 0.8840 - val_loss: 0.1757 - val_accuracy: 0.9505
Epoch 2/2000
112/112 [==============================] - ETA: 0s - loss: 0.0404 - accuracy: 0.9902
Epoch 2: saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
112/112 [==============================] - 17s 149ms/step - loss: 0.0404 - accuracy: 0.9902 - val_loss: 0.0053 - val_accuracy: 0.9983
Epoch 3/2000
112/112 [==============================] - ETA: 0s - loss: 0.0020 - accuracy: 1.0000
Epoch 3: saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
112/112 [==============================] - 16s 142ms/step - loss: 0.0020 - accuracy: 1.0000 - val_loss: 1.6971e-04 - val_accuracy: 1.0000
Epoch 4/2000
112/112 [=================

# Evaluate Model

In [7]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_acc}')

38/38 [==============================] - 3s 73ms/step - loss: 0.0140 - accuracy: 0.9958
Validation Loss: 0.014002685435116291, Validation Accuracy: 0.9958088994026184


# Convert to TFLite

In [8]:
model.save(model_save_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print("TFLite model saved.")

INFO:tensorflow:Assets written to: C:\Users\PC\AppData\Local\Temp\tmp4dc9dq2f\assets


INFO:tensorflow:Assets written to: C:\Users\PC\AppData\Local\Temp\tmp4dc9dq2f\assets


TFLite model saved.


# Test Inference

In [9]:
# Test Inference using Keras model
# Load the saved Keras model
loaded_model = tf.keras.models.load_model(model_save_path)

# Test on all test samples
predictions = loaded_model.predict(X_test, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

# Calculate overall accuracy
correct = np.sum(predicted_classes == y_test)
total = len(y_test)
print(f"Overall Test Accuracy: {correct}/{total} = {correct/total*100:.2f}%\n")

# Show per-class accuracy
print("Per-class Performance:")
# Use the actual number of classes in the data
actual_num_classes = int(max(y_test.max(), y_train.max()) + 1)
for i in range(min(NUM_CLASSES, actual_num_classes)):
    class_mask = y_test == i
    if np.sum(class_mask) > 0:
        class_correct = np.sum((predicted_classes == i) & (y_test == i))
        class_total = np.sum(class_mask)
# Show confusion matrix
print("\n" + "="*50)
print("Confusion Matrix:")
print("="*50)
from sklearn.metrics import confusion_matrix
# Get actual number of classes in the data
actual_num_classes = int(max(y_test.max(), y_train.max()) + 1)
cm = confusion_matrix(y_test, predicted_classes)
print(f"\n{'':10}", end="")
for i in range(min(len(keypoint_sequence_classifier_labels), actual_num_classes)):
    label = keypoint_sequence_classifier_labels[i] if i < len(keypoint_sequence_classifier_labels) else f"Class_{i}"
    print(f"{label:10}", end="")
print()
for i in range(min(len(keypoint_sequence_classifier_labels), actual_num_classes)):
    label = keypoint_sequence_classifier_labels[i] if i < len(keypoint_sequence_classifier_labels) else f"Class_{i}"
    print(f"{label:10}", end="")
# Show sample predictions from each class
print("\n" + "="*50)
print("Sample Predictions:")
print("="*50)
actual_num_classes = int(max(y_test.max(), y_train.max()) + 1)
for class_idx in range(min(NUM_CLASSES, actual_num_classes)):

# Show sample predictions from each class
 print("\n" + "="*50)
print("Sample Predictions:")
print("="*50)
for class_idx in range(NUM_CLASSES):
    class_samples = np.where(y_test == class_idx)[0]
    if len(class_samples) > 0:
        sample_idx = class_samples[0]
        pred_class = predicted_classes[sample_idx]
        confidence = predictions[sample_idx][pred_class] * 100
        print(f"\nActual: {keypoint_sequence_classifier_labels[class_idx]}")
        print(f"Predicted: {keypoint_sequence_classifier_labels[pred_class]} (confidence: {confidence:.2f}%)")
        if pred_class == class_idx:
            print("✓ Correct")
        else:
            print("✗ Wrong")

Overall Test Accuracy: 1188/1193 = 99.58%

Per-class Performance:

Confusion Matrix:

          全部        ---       多謝        失望        來         byebye    
全部        ---       多謝        失望        來         byebye    
Sample Predictions:






Sample Predictions:

Actual: 全部
Predicted: 全部 (confidence: 100.00%)
✓ Correct

Actual: 多謝
Predicted: 多謝 (confidence: 100.00%)
✓ Correct

Actual: 失望
Predicted: 失望 (confidence: 99.99%)
✓ Correct

Actual: 來
Predicted: 來 (confidence: 100.00%)
✓ Correct

Actual: byebye
Predicted: byebye (confidence: 100.00%)
✓ Correct
